# Safety-circuits — §9 editing suite (portable: Kaggle **and** Google Colab)

One cell, runs on **either** platform — it auto-detects Kaggle vs Colab and sets the HF token
source + output dir accordingly. Change only `MODEL`, then run. Runs the full editing study + every
Tier 1/2 extension for one model in a single pass.

**One-time setup (both):** enable a **GPU T4** runtime, and add an `HF_TOKEN` secret (HF account must
have accepted the gemma-3 / llama-3.2 gated terms).
- **Kaggle:** Settings → Accelerator = GPU T4, Internet = On; Add-ons → Secrets → `HF_TOKEN`.
- **Colab:** Runtime → Change runtime type → T4 GPU; 🔑 (Secrets) → add `HF_TOKEN`, enable notebook access.

**Where results go:** Kaggle → `/kaggle/working/editing/<MODEL>/` (download from the version Output).
Colab → mounts Drive and writes to `MyDrive/safety-circuits-editing/editing/<MODEL>/` (survives disconnects).

**Colab free caveats:** idle/disconnect ~90 min and dynamic GPU quota — a full run is ~3–4 h, so do
**one model per session**. `SC_SKIP_EXISTING=1` is set, so re-launching resumes (skips finished models).
First time, smoke-test with `SC_EDIT_STEPS=50` to confirm everything writes.

**Ethics:** the benign substance-unlock test (T1.1b) trains/evaluates on benign content only; we do not
train on or elicit harmful generation (see `RESEARCH_PLAN.md` §9 ethics).

In [ ]:
import os, pathlib, subprocess, sys

MODEL = "gemma3-1b"   # <-- the only knob. one of:
                      # gemma3-1b, qwen3, qwen2.5, qwen2-1.5b, qwen1.5-1.8b,
                      # gemma1-2b, gemma2-2b, llama3.2-1b, llama3-3b

# ---- platform auto-detect: HF token source + output dir (Kaggle / Colab / Lightning|local) ----
ON_KAGGLE = os.path.isdir("/kaggle/working")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except Exception:
    ON_COLAB = False

if not os.environ.get("HF_TOKEN"):                    # try platform secret stores, else warn
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except Exception:
            print("WARNING: no HF_TOKEN found; set os.environ['HF_TOKEN'] (gated gemma/llama need it).")

if os.environ.get("SC_OUT"):
    pass                                              # explicit override wins
elif ON_KAGGLE:
    os.environ["SC_OUT"] = "/kaggle/working"          # persists in the version Output
elif ON_COLAB:
    from google.colab import drive                    # persist to Drive (survives disconnects)
    drive.mount("/content/drive")
    os.environ["SC_OUT"] = "/content/drive/MyDrive/safety-circuits-editing"
else:                                                 # Lightning AI / local / generic Linux box
    os.environ["SC_OUT"] = os.path.expanduser("~/safety-circuits-editing")
print("platform:", "Kaggle" if ON_KAGGLE else ("Colab" if ON_COLAB else "Lightning/local"),
      "| SC_OUT =", os.environ["SC_OUT"])

# ---- core editing run (validated config) ----
os.environ["SC_MODELS"]               = MODEL
os.environ["SC_SKIP_EXISTING"]        = "1"        # resume: skip models already done in SC_OUT
os.environ["SC_EDIT_STEPS"]           = "600"
os.environ["SC_EDIT_RANK"]            = "16"
os.environ["SC_EDIT_LR"]              = "5e-4"
os.environ["SC_EDIT_HEADCOUNTS"]      = "1,3,5,10"
os.environ["SC_EDIT_METHODS"]         = "steering,lora"

# ---- Tier 1/2 extensions: everything ON ----
os.environ["SC_DO_GENERALIZATION"]    = "1"        # long-form + per-category + toxicity        (T1.1)
os.environ["SC_DO_DIRSHIFT"]          = "1"        # refusal-direction rotation (mechanism)      (T2.6/2.7)
os.environ["SC_DO_HARDENING"]         = "1"        # comply->refuse safety re-patch round-trip   (T2.5)
os.environ["SC_DO_MINIMAL_SWEEP"]     = "1"        # smallest clean edit                         (T1.2)
os.environ["SC_EDIT_MINIMAL_RANKS"]   = "8,16"     # trim the sweep grid (2x2 = 4 trains)
os.environ["SC_EDIT_MINIMAL_STEPS"]   = "300,600"
os.environ["SC_DO_BENIGN_SUBSTANCE"]  = "1"        # weapon-free substance-unlock test           (T1.1b)
os.environ["SC_EDIT_MAX_TARGET_TOKENS"] = "128"   # long benign target length

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- deps + drop ABI-mismatched torchaudio/torchvision (unused; would crash the import) ----
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformer-lens>=2.0.0", "transformers>=4.40", "datasets>=2.18",
    "accelerate>=0.27", "einops>=0.7", "seaborn", "tqdm", "pyyaml", "jaxtyping",
    "peft>=0.10"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio", "torchvision"],
               check=False)

# ---- pull latest repo (/tmp works on both platforms) ----
repo = pathlib.Path("/tmp/safety-circuits")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/pra-nav-04/safety-circuits.git", str(repo)], check=True)
sys.path.insert(0, str(repo / "src"))

# ---- purge stale cached modules so the freshly pulled code loads on a warm kernel ----
for _m in [m for m in list(sys.modules)
           if m.split(".")[0] in ("safety_circuits", "transformer_lens", "transformers",
                                   "torchaudio", "torchvision")]:
    del sys.modules[_m]

import runpy
runpy.run_path(str(repo / "kaggle" / "run_edit_experiment.py"), run_name="__main__")